In [1]:
import yfinance as yf
from datetime import datetime

def parse_option_symbol(option_str):
    """
    Parse an option string like 'GOOG 16JAN26 150 C' into components.
    Returns dict with underlying, expiry (YYYY-MM-DD), strike, and type.
    """
    parts = option_str.strip().split()
    if len(parts) != 4:
        raise ValueError(f"Invalid option format: {option_str}")

    underlying, date_str, strike, opt_type = parts
    expiry = datetime.strptime(date_str, "%d%b%y").strftime("%Y-%m-%d")

    return {
        "Underlying": underlying.upper(),
        "Expiry": expiry,
        "Strike": float(strike),
        "Type": opt_type.upper()
    }


def get_option_info(option_str, as_of_date=None):
    """
    Universal option info retriever — works with option strings like:
    'GOOG 16JAN26 150 C' or 'AMZN 17JUN27 100 C'
    """
    # Parse the string
    parsed = parse_option_symbol(option_str)
    underlying = parsed["Underlying"]
    expiry = parsed["Expiry"]
    strike = parsed["Strike"]
    call_put = parsed["Type"]

    # Fetch ticker
    ticker = yf.Ticker(underlying)

    # Try to get currency
    try:
        info = ticker.info
        currency = info.get("currency", "USD")
    except Exception:
        currency = "USD"

    # Try to get option chain
    try:
        chain = ticker.option_chain(expiry)
    except Exception as e:
        print(f"Could not fetch option chain for {underlying} {expiry}: {e}")
        return None

    df = chain.calls if call_put == "C" else chain.puts
    row = df[df['strike'] == strike]
    if row.empty:
        print(f"Option not found for {option_str}")
        return None

    date_retrieved = as_of_date or datetime.now().strftime("%Y-%m-%d")

    return {
        "Option": option_str,
        "Underlying": underlying,
        "Expiry": expiry,
        "Strike": strike,
        "Type": call_put,
        "Date": date_retrieved,
        "Currency": currency,
        "LastPrice": row["lastPrice"].values[0],
        "Bid": row["bid"].values[0],
        "Ask": row["ask"].values[0],
        "ImpliedVolatility": row["impliedVolatility"].values[0],
        "ContractSymbol": row["contractSymbol"].values[0],
    }

options = [
    "GOOG 16JAN26 150 C",
    "AMZN 17JUN27 100 C"
]

for opt in options:
    data = get_option_info(opt)
    print(data)

{'Option': 'GOOG 16JAN26 150 C', 'Underlying': 'GOOG', 'Expiry': '2026-01-16', 'Strike': 150.0, 'Type': 'C', 'Date': '2025-11-16', 'Currency': 'USD', 'LastPrice': 130.4, 'Bid': 126.3, 'Ask': 129.6, 'ImpliedVolatility': 0.8247087841796874, 'ContractSymbol': 'GOOG260116C00150000'}
{'Option': 'AMZN 17JUN27 100 C', 'Underlying': 'AMZN', 'Expiry': '2027-06-17', 'Strike': 100.0, 'Type': 'C', 'Date': '2025-11-16', 'Currency': 'USD', 'LastPrice': 148.22, 'Bid': 142.65, 'Ask': 145.45, 'ImpliedVolatility': 0.6551853329467775, 'ContractSymbol': 'AMZN270617C00100000'}


In [19]:
import pandas as pd
from datetime import timedelta


def parse_option_symbol(option_str):
    """
    Parse an option string like 'GOOG 16JAN26 150 C' into components.
    """
    parts = option_str.strip().split()
    if len(parts) != 4:
        raise ValueError(f"Invalid option format: {option_str}")

    underlying, date_str, strike, opt_type = parts
    expiry = datetime.strptime(date_str, "%d%b%y").strftime("%Y-%m-%d")

    return {
        "Underlying": underlying.upper(),
        "Expiry": expiry,
        "Strike": float(strike),
        "Type": opt_type.upper(),
    }

def get_option_info(option_str, start_date, get_exchange_rates):
    parsed = parse_option_symbol(option_str)
    underlying = parsed["Underlying"]
    expiry = parsed["Expiry"]
    strike = parsed["Strike"]
    call_put = parsed["Type"]

    ticker = yf.Ticker(underlying)

    # --- Get currency from ticker.info ---
    try:
        info = ticker.info
        currency = info.get("currency", "USD")
    except Exception:
        currency = "USD"

    # --- Try to get the option chain ---
    try:
        chain = ticker.option_chain(expiry)
    except Exception as e:
        print(f"Could not fetch option chain for {underlying} {expiry}: {e}")
        return None

    df = chain.calls if call_put.upper() == "C" else chain.puts
    row = df[df["strike"] == strike]
    if row.empty:
        print(f"Option not found for {option_str}")
        return None

    # --- Get today's date ---
    today = datetime.now().strftime("%Y-%m-%d")

    # --- Base data (latest market value) ---
    latest_close = row["lastPrice"].values[0]
    date_retrieved = pd.to_datetime(today)

    # --- FX conversion to CHF if needed ---
    if currency != "CHF":
        exchange_df = get_exchange_rates(currency, "CHF", date_retrieved)
        if not exchange_df.empty:
            rate = exchange_df["Rate"].iloc[-1]
        else:
            rate = 1.0
    else:
        rate = 1.0

    close_chf = latest_close * rate

    start_date_dt = pd.to_datetime(start_date)
    end_date_dt = pd.to_datetime(today)
    date_range = pd.date_range(start=start_date_dt, end=end_date_dt, freq="D")

    df_data = pd.DataFrame({
        "Date": date_range,
        "Symbol": option_str,
        "Currency": currency,
        "Close_CHF": close_chf,
        "Close_CHF_constant": close_chf,   # same value now, can adjust later if needed
        "Dividends_CHF": 0.0,
        "Stock Splits": 0.0,
    })

    df_data = df_data.ffill().bfill()

    df_data = df_data[["Date", "Dividends_CHF", "Symbol", "Close_CHF", "Stock Splits", "Close_CHF_constant"]]

    return df_data

def get_exchange_rates(base_currency, target_currency='CHF', start_date='2021-01-01'):
    # Fetch historical data for the currency pair
    currency_pair = f'{base_currency}{target_currency}=X'
    ticker = yf.Ticker(currency_pair)
    exchange_df = ticker.history(start=start_date, interval='1d')

    exchange_df = exchange_df[['Close']]  # Use 'Close' as the exchange rate
    exchange_df.rename(columns={'Close': 'Rate'}, inplace=True)
    exchange_df.index = exchange_df.index.date  # Convert index to date only
    exchange_df.index.name = 'Date'
    return exchange_df

def get_exchange_rates(base_currency, target_currency='CHF', start_date='2021-01-01',
                       max_backdays=5, verbose=False):
    """
    Fetch historical exchange rates base_currency -> target_currency as 'Rate'.
    If history(start=...) returns empty, step start_date back one day at a time
    up to max_backdays and retry. If still empty, try period='5d' as a final fallback.

    Returns DataFrame indexed by Date (date objects) with column 'Rate'.
    """
    # Normalize currencies and basic checks
    base_currency = str(base_currency).upper()
    target_currency = str(target_currency).upper()
    if base_currency == target_currency:
        # trivial rate = 1.0 for the requested date range: try to return a small series
        today = pd.Timestamp.today().normalize().date()
        return pd.DataFrame({"Rate": [1.0]}, index=pd.Index([today], name="Date"))

    currency_pair = f"{base_currency}{target_currency}=X"
    ticker = yf.Ticker(currency_pair)

    # Normalize start_date to a pandas Timestamp (date only)
    start_ts = pd.to_datetime(start_date)
    # If the provided start is in the future, set it to today
    today_ts = pd.Timestamp.today().normalize()
    if start_ts > today_ts:
        if verbose:
            print(f"start_date {start_ts.date()} is in the future — using today {today_ts.date()} as starting point")
        start_ts = today_ts

    # Try the initial start and then step back one day at a time
    exchange_df = pd.DataFrame()
    attempts = 0
    current_start = start_ts

    while attempts <= max_backdays:
        if verbose:
            print(f"Attempt {attempts+1}: requesting history(start={current_start.date()}) for {currency_pair}")
        try:
            exchange_df = ticker.history(start=current_start.strftime("%Y-%m-%d"), interval="1d")
        except Exception as e:
            if verbose:
                print(f"history call raised: {e}")
            exchange_df = pd.DataFrame()

        if not exchange_df.empty:
            if verbose:
                print(f"Got data on attempt {attempts+1} starting {current_start.date()}")
            break

        # step back one calendar day and retry
        current_start = current_start - timedelta(days=1)
        attempts += 1

    # Final fallback: try recent period if still empty
    if exchange_df.empty:
        if verbose:
            print(f"No data after {attempts} back-steps — trying period='5d' fallback for {currency_pair}")
        try:
            exchange_df = ticker.history(period="5d", interval="1d")
        except Exception as e:
            if verbose:
                print(f"period fallback raised: {e}")
            exchange_df = pd.DataFrame()

    # If still empty, return empty DataFrame
    if exchange_df.empty:
        if verbose:
            print(f"⚠️ No FX data available for {currency_pair}")
        return exchange_df

    # Normalize and return
    if "Close" not in exchange_df.columns and "close" in exchange_df.columns:
        exchange_df.rename(columns={"close": "Close"}, inplace=True)

    result = exchange_df[["Close"]].copy()
    result.rename(columns={"Close": "Rate"}, inplace=True)
    # make index plain date objects and name it 'Date'
    result.index = pd.to_datetime(result.index).date
    result.index.name = "Date"
    return result

options = [
    "GOOG 16JAN26 150 C",
    "AMZN 17JUN27 100 C"
]

for opt in options:
    data = get_option_info(opt, '2021-01-01', get_exchange_rates)


$USDCHF=X: possibly delisted; no price data found  (1d 2025-11-16 -> 2025-11-16)
$USDCHF=X: possibly delisted; no price data found  (1d 2025-11-16 -> 2025-11-16)


In [20]:
print(data)
start_date='2021-01-01'
exchange_df = yf.Ticker("USDCHF=X").history(start=start_date, interval='1d')

print(exchange_df)

           Date  Dividends_CHF              Symbol   Close_CHF  Stock Splits  \
0    2021-01-01            0.0  AMZN 17JUN27 100 C  117.649623           0.0   
1    2021-01-02            0.0  AMZN 17JUN27 100 C  117.649623           0.0   
2    2021-01-03            0.0  AMZN 17JUN27 100 C  117.649623           0.0   
3    2021-01-04            0.0  AMZN 17JUN27 100 C  117.649623           0.0   
4    2021-01-05            0.0  AMZN 17JUN27 100 C  117.649623           0.0   
...         ...            ...                 ...         ...           ...   
1776 2025-11-12            0.0  AMZN 17JUN27 100 C  117.649623           0.0   
1777 2025-11-13            0.0  AMZN 17JUN27 100 C  117.649623           0.0   
1778 2025-11-14            0.0  AMZN 17JUN27 100 C  117.649623           0.0   
1779 2025-11-15            0.0  AMZN 17JUN27 100 C  117.649623           0.0   
1780 2025-11-16            0.0  AMZN 17JUN27 100 C  117.649623           0.0   

      Close_CHF_constant  
0           